# torch.compile with Llama 3.1 8B – Inference Optimization

**What this notebook does:** We load Meta's Llama 3.1 8B model and compare inference speed with and without `torch.compile`. This PyTorch 2.0 feature compiles your model into optimized kernels for faster execution.

**Concepts you'll learn:**
- Loading a large language model (LLM) with 4-bit quantization (to fit on a single GPU)
- Measuring inference latency (time to generate tokens)
- Using `torch.compile` to speed up generation
- Why "warmup" runs matter for compiled models

## 1. Environment Setup (Colab)

Run these cells to install CUDA tools, clone the repo, and install Python packages.  
*Skip if running locally and dependencies are already installed.*

In [ ]:
# Install CUDA toolkit (needed for GPU support in Colab)
!apt-get install -y cuda-toolkit

In [ ]:
# Clone the repo and navigate to this folder
# %cd = change directory (Jupyter magic command)
!git clone https://github.com/HamidrezaGh/llm-performance.git
%cd llm-performance/02-inference-optimization
!ls

Cloning into 'llm-performance'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 104 (delta 45), reused 62 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 29.58 KiB | 2.96 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/llm-performance/llm-performance/llm-performance/llm-performance/llm-performance/02-inference-optimization
01-torch-compile-llama8b  03-continuous-batching    04-70b-multi-gpu-serving
02-triton-in-llama	  03-torch-compile-llama8b


In [10]:
# Install required packages:
# - transformers: Hugging Face library for loading LLMs
# - torch: PyTorch (deep learning framework)
# - accelerate: multi-GPU / quantization utilities
# - bitsandbytes: 4-bit quantization (reduces memory so 8B model fits on one GPU)
!pip install -q transformers torch accelerate bitsandbytes

## 2. Load the Model

We use **4-bit quantization** (BitsAndBytes) so the 8B-parameter model fits in ~5GB VRAM instead of ~16GB.  
You need a **Hugging Face token** with access to Meta-Llama-3.1-8B. Add it as a Colab secret named `HF_TOKEN`.

In [17]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Log in to Hugging Face (required for gated models like Llama)
# userdata.get('HF_TOKEN') reads your token from Colab Secrets
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

# Model name - Meta's instruction-tuned 8B Llama
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 4-bit quantization: store weights in 4 bits instead of 16 (fp16)
# - nf4: normalized float 4-bit (good for LLM weights)
# - bfloat16 compute: do math in bf16 for speed
# - double quant: quantize the quantization constants (saves more memory)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer (converts text <-> token IDs) and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",  # Automatically put layers on GPU(s)
    torch_dtype=torch.bfloat16
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded successfully!


## 3. Baseline: Generate Without torch.compile

We time how long it takes to generate 50 new tokens.  
`torch.cuda.Event` gives accurate GPU timing (better than `time.time()` for CUDA).

In [18]:
prompt = "The future of AI is"
# Tokenize: convert text to token IDs, return as PyTorch tensors, move to GPU
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# CUDA events for precise GPU timing (more accurate than Python's time.time)
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
with torch.no_grad():  # Disable gradient tracking (saves memory during inference)
    outputs = model.generate(**inputs, max_new_tokens=50)
end.record()
torch.cuda.synchronize()  # Wait for GPU to finish before reading the time

latency_ms = start.elapsed_time(end)
print(f"Baseline latency: {latency_ms:.2f} ms")
print(tokenizer.decode(outputs[0]))  # Convert token IDs back to text

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Baseline latency: 6061.38 ms
<|begin_of_text|>The future of AI is here, and it's already changing the way we work, live, and interact with each other. Artificial intelligence (AI) is a rapidly evolving field that's transforming industries, revolutionizing products, and redefining the way we do things.
In


## 4. torch.compile: Same Model, Optimized

`torch.compile()` traces the model and compiles it into fused kernels.  
- `mode="reduce-overhead"`: optimizes for Python overhead (good for small batches)  
- `mode="max-autotune"`: tries many kernel variants for max speed (slower compile, faster run)

**Note:** First run after compile is slow (compilation happens). Later runs are timed.

In [19]:
# Wrap the model with torch.compile - PyTorch will optimize the forward pass
compiled_model = torch.compile(model, mode="reduce-overhead")  # or "max-autotune"

start.record()
with torch.no_grad():
    outputs_comp = compiled_model.generate(**inputs, max_new_tokens=50)
end.record()
torch.cuda.synchronize()

comp_latency_ms = start.elapsed_time(end)
print(f"Compiled latency: {comp_latency_ms:.2f} ms")
print(f"Speedup: {latency_ms / comp_latency_ms:.2f}×")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Compiled latency: 6068.17 ms
Speedup: 1.00×


## 5. Longer Benchmark: 200 Tokens, Greedy Decoding

- **Warmup:** The first call to `compiled_model.generate()` triggers compilation. We run it once before timing.
- **do_sample=False:** Greedy decoding (pick highest-probability token) – deterministic, faster.
- **TPS:** Tokens per second – throughput metric.

In [20]:
import time
import torch

# Reuse model, compiled_model, tokenizer from above

prompt = "Explain the difference between inference and training in large language models in detail."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Warmup: first compiled run triggers JIT compilation - don't time this!
print("Warmup run...")
with torch.no_grad():
    _ = compiled_model.generate(**inputs, max_new_tokens=100)
torch.cuda.synchronize()

# Timed run - baseline (non-compiled)
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
with torch.no_grad():
    outputs_baseline = model.generate(**inputs, max_new_tokens=200, do_sample=False)
end.record()
torch.cuda.synchronize()
baseline_latency = start.elapsed_time(end)
print(f"Baseline latency (200 tokens): {baseline_latency:.2f} ms")

# Timed run - compiled (after warmup)
start.record()
with torch.no_grad():
    outputs_compiled = compiled_model.generate(**inputs, max_new_tokens=200, do_sample=False)
end.record()
torch.cuda.synchronize()
compiled_latency = start.elapsed_time(end)
print(f"Compiled latency (200 tokens): {compiled_latency:.2f} ms")
print(f"Speedup: {baseline_latency / compiled_latency:.2f}×")

# TPS = tokens per second (throughput)
tokens_generated = 200
print(f"Baseline TPS: {tokens_generated / (baseline_latency / 1000):.1f}")
print(f"Compiled TPS: {tokens_generated / (compiled_latency / 1000):.1f}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Warmup run...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Baseline latency (200 tokens): 18753.08 ms
Compiled latency (200 tokens): 18542.37 ms
Speedup: 1.01×
Baseline TPS: 10.7
Compiled TPS: 10.8


## 6. Long Generation with Sampling

- **do_sample=True:** Random sampling instead of greedy – more diverse outputs.
- **temperature=0.7:** Higher = more random; lower = more deterministic.
- **top_p=0.9:** Nucleus sampling – consider only top tokens whose cumulative probability ≤ 0.9.

*Note: torch.compile gains vary by workload. Autoregressive generation (one token at a time) often sees smaller speedups than batched forward passes.*

In [21]:
import torch
import time

# Reuse model, compiled_model, tokenizer from above

# Longer generation (400 tokens) with sampling for more natural text
prompt = "Write a detailed 400-word essay on why scaling laws matter for AI safety."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Warmup both models (compiled needs it; baseline gets same treatment for fair comparison)
print("Warmup runs...")
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
    _ = compiled_model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
torch.cuda.synchronize()

# Baseline timing
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
with torch.no_grad():
    outputs_baseline = model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
end.record()
torch.cuda.synchronize()
baseline_latency = start.elapsed_time(end)
baseline_tokens = len(outputs_baseline[0]) - len(inputs.input_ids[0])  # New tokens only
baseline_tps = baseline_tokens / (baseline_latency / 1000)

print(f"Baseline: {baseline_latency:.2f} ms | {baseline_tps:.1f} TPS")

# Compiled timing
start.record()
with torch.no_grad():
    outputs_compiled = compiled_model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
end.record()
torch.cuda.synchronize()
compiled_latency = start.elapsed_time(end)
compiled_tokens = len(outputs_compiled[0]) - len(inputs.input_ids[0])
compiled_tps = compiled_tokens / (compiled_latency / 1000)

print(f"Compiled: {compiled_latency:.2f} ms | {compiled_tps:.1f} TPS")
print(f"Speedup: {baseline_latency / compiled_latency:.2f}× | TPS gain: {compiled_tps / baseline_tps:.2f}×")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Warmup runs...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Baseline: 36458.14 ms | 11.0 TPS
Compiled: 51144.92 ms | 7.8 TPS
Speedup: 0.71× | TPS gain: 0.71×
